In [0]:
from datetime import datetime
import pytz
from pyspark.sql import functions as F

In [0]:
fuso_br = pytz.timezone('America/Sao_Paulo')
agora = datetime.now(fuso_br)
particao_hoje = f"ano={agora.year}/mes={agora.month:02d}/dia={agora.day -1:02d}"

In [0]:
meu_storage = "stgbbb"
spark.conf.set(
    f"fs.azure.account.key.{meu_storage}.dfs.core.windows.net",
    "xzhcoxUXRj+GjVy8wOc3wladWad/Dm+OgQ5Lpj6U0RKd8K0+n18AJz12D3FmA/LTGP8SsbqSRybt+AStxQY00w=="
)

meu_container_destino = "silver"
 
meu_container_origem = "bronze"
caminho_origem = f"abfss://{meu_container_origem}@{meu_storage}.dfs.core.windows.net/"

In [0]:
tabelas_silver_config = {
    "empresas": {
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Empresas*", 
        "destino": "silver/receita/empresas",
        "particao": None, 
        "colunas_data": [] 
    },
    "estabelecimento": {
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Estabelecimentos*",
        "destino": "silver/receita/estabelecimentos",
        "particao": "uf", 
        "colunas_data": ["data_situacao_cadastral", "data_inicio_atividade", "data_situacao_especial"]
    },
    "socios": {
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Socios*",
        "destino": "silver/receita/socios",
        "particao": None,
        "colunas_data": ["data_entrada_sociedade"]
    },
    "dados_simples": {
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Simples*", 
        "destino": "silver/receita/dados_simples",
        "particao": None,
        "colunas_data": ["data_opcao_simples", "data_exclusao_simples", "data_opcao_mei", "data_exclusao_mei"]
    },
    "cnaes": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Cnaes*", 
        "destino": "silver/receita/auxiliar_cnaes", "particao": None, "colunas_data": [] 
    },
    "municipios": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Municipios*", 
        "destino": "silver/receita/auxiliar_municipios", "particao": None, "colunas_data": [] 
    },
    "paises": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Paises*", 
        "destino": "silver/receita/auxiliar_paises", "particao": None, "colunas_data": [] 
    },
    "motivos": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Motivos*", 
        "destino": "silver/receita/auxiliar_motivos", "particao": None, "colunas_data": [] 
    },
    "natureza_juridica": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Naturezas*", 
        "destino": "silver/receita/auxiliar_natureza_juridica", "particao": None, "colunas_data": [] 
    },
    "qualificacao_socio": { 
        "origem": f"{caminho_origem}cnpj/{particao_hoje}/Qualificacoes*", 
        "destino": "silver/receita/auxiliar_qualificacao_socio", "particao": None, "colunas_data": [] 
    }
}

In [0]:
for nome_tabela, config in tabelas_silver_config.items():
    print(f"Processando Silver: {nome_tabela}...")
    
    df = spark.read.format("parquet").load(config["origem"])

    # display(df)
    
#     # 2. Tratamento: Converter datas (formato YYYYMMDD padrão Receita para DateType)
#     # No seu schema original tudo é StringType, aqui convertemos para Date
#     for col_data in config["colunas_data"]:
#         # Verifica se a data não é '0' ou '00000000' que as vezes vem sujeira
#         df = df.withColumn(col_data, 
#                            F.when(F.col(col_data) == '00000000', None)
#                             .when(F.col(col_data) == '0', None)
#                             .otherwise(F.to_date(F.col(col_data), 'yyyyMMdd')))
    
#     # 3. Tratamento Opcional: Converter Capital Social para Double (se existir na tabela)
#     if "capital_social" in df.columns:
#         # Troca vírgula por ponto (padrão Brasil -> US)
#         df = df.withColumn("capital_social", F.regexp_replace("capital_social", ",", ".").cast("double"))

#     # 4. Escrita na Silver (Particionada)
#     writer = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    
#     if config["particao"]:
#         print(f"   -> Particionando por: {config['particao']}")
#         writer = writer.partitionBy(config["particao"])
    
#     writer.save(config["destino"])
#     print(f"   -> Salvo em: {config['destino']}")

# print("=== Fim da Carga Silver ===")